In [2]:
import os

In [3]:
%pwd

'c:\\Users\\smart\\OneDrive\\Documents\\End to End Production Grade industry ready AI projects\\AI Powered Content Summarization\\Content-Summarization\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\smart\\OneDrive\\Documents\\End to End Production Grade industry ready AI projects\\AI Powered Content Summarization\\Content-Summarization'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from content_summarization.constants import *
from content_summarization.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [9]:
import os
import urllib.request as request
import zipfile

from content_summarization.logging import logger
from content_summarization.utils.common import get_size

In [10]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [14]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-04-25 22:37:33,176]: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-25 22:37:33,180]: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-25 22:37:33,182]: INFO: common: created directory at: artifacts]
[2026-04-25 22:37:33,184]: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-25 22:37:37,113]: INFO: 1434958058: artifacts/data_ingestion/samsumdata.zip download! with following info: 
Connection: close
Content-Length: 23627009
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "2e7409b328d118a1d37018be788babf8bf9640386387da766ac100e96efa3b93"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: DC82:30FC0C:6F5D62:86AA1F:69ED7A86
Accept-Ranges: bytes
Date: Sun, 26 Apr 2026 02:37:58 GMT
Via: 1.1 varnish
X-Served-By: cache-yyz4524-YYZ